In [11]:
import pandas as pd
import requests, json
from pathlib import Path
from datetime import datetime

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

FILE_REAL = BASE_PATH / "data" / "raw" / "EXPORT_REAL.XLSX"
FILE_COMP = BASE_PATH / "data" / "raw" / "EXPORT_COMPROMIS.XLSX"
FILE_SITE = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

URL_APPS_SCRIPT = "https://script.google.com/macros/s/AKfycbyeLMGPlTdfRNFBIb-Y_N_ZQgg6rz0r5IcW5QuNpWWSb0QRrCw13h6-WVDiBaN2jkCyjg/exec"

# ======================================================
# CONSTANTES
# ======================================================

# Tarefas que aparecem em TODOS os tipos de projeto
TAREFAS_LOJAS_NOVAS = [
    "Construtora", "Gerenciadora", "Alvo K - Utilities", "Persianas", "Luminária",
    "Ar Condicionado - AC", "Linha Branca", "Projetos", "Comunicação Visual",
    "Mobiliário", "Montagem", "Frete", "Gaveta", "Permits", "CDU", "Capitalização",
    "Provisionamento", "Preservação", "Transferência"
]

# Tarefas para projetos que NÃO são Lojas Novas
# Substitui "Preservação" por "Reforma de Loja"
TAREFAS_OUTROS = [
    "Construtora", "Gerenciadora", "Alvo K - Utilities", "Persianas", "Luminária",
    "Ar Condicionado - AC", "Linha Branca", "Projetos", "Comunicação Visual",
    "Mobiliário", "Montagem", "Frete", "Gaveta", "Permits", "CDU", "Capitalização",
    "Provisionamento", "Reforma de Loja", "Transferência"
]

# Tipos que seguem as tarefas de Lojas Novas
TIPO_LOJAS_NOVAS = "Lojas Novas"

# Materiais de Mobiliário (lista completa fornecida pela equipe)
MATERIAIS_MOBILIARIO = {
    20398,20534,20581,20582,20583,20584,20585,20586,20587,20588,
    20589,20590,20591,20592,20593,20594,20595,20596,20597,20598,
    20599,20600,20601,20605,20606,20607,20609,20610,20611,20612,
    20613,20614,20615,20616,20617,20618,20619,20620,20621,20622,
    20623,20624,20625,20626,20629,20630,20631,20632,20633,20773,
    20774,20775,20777,20778,20781,20783,20784,20786,20788,20789,
    20790,20792,20793,20794,20795,20796,20797,20798,20799,20801,
    20806,20807,20808,20809,20814,20815,20816,20822,24742,24743,
    24745,24775,24815,24832,25087,25088,25089,25090,25132,25133,
    25427,25429,25430,25431,25432,25696,25697,25891,25892,25896,
    25898,25899,25902,25903,25906,25907,25932,25933,25934,25946,
    25949,25950,25951,25954,25955,25956,25977,26009,26010,26011,
    26012,26014,26016,26017,26018,26019,26020,26021,26022,26023,
    26024,26025,26026,26027,26028,26029,26030,26031,26032,26033,
    26035,26036,26046,26047,26048,26049,26050,26051,26052,26053,
    26054,26055,26056,26057,26058,26059,26060,26061,26062,26063,
    26161,26162,26163,26164,26165,26176,26177,26178,26180,26181,
    26182,26183,26184,26185,26186,26187,26188,26189,26190,26191,
    26192,26193,26194,26195,26196,26197,26198,26199,26210,26211,
    26232,26233,26234,26235,26242,26243,26244,26246,26247,26248,
    26257,26258,26259,26317,26326,26328,26330,26331,26332,26340,
    26345,26374,26414,26415,26433,26531,26611,26645,26664,26710,
    26711,27880,27881,27882,27883,29464,29718,29719,29720,29721,
    29724,29728,29731,29734,29735,29736,29737,29740,29741,29758,
    29759,29762,29763,29767,29786,29787,29788,29789,29790,29791,
    29792,29793,29800,29801,29802,29803,29804,29805,29806,29808,
    29809,30061,30062,30063,30398,30520,30526,30527,30531,30536,
    32101,32437,32482,32488,32527,32550,32552,32554,32555,32556,
    32557,32570,32571,32572,32573,32574,32575,32577,32578,32579,
    32580,32581,32582,32583,32584,32585,32586,32587,32588,32589,
    32590,32591,32592,32593,32598,32600,32601,32602,32603,32604,
    32605,32606,32607,32608,32611,32614,32615,32670,32677,32698,
    32733,32750,32753,32754,32761
}

# Gaveta: material 24780 — vem no mesmo pedido do Mobiliário mas é nota separada
# Montagem e Frete: pedidos totalmente distintos
MATERIAIS_GAVETA    = {24780}
MATERIAIS_MONTAGEM  = {24935}
MATERIAIS_FRETE     = {20272}

REGRAS_MATERIAIS = {
    "Construtora":        {20102},
    "Gerenciadora":       {32432},
    "Permits":            {40143, 39160, 25044, 24822, 24769, 40195, 20311, 20702, 40069, 40252},
    "Linha Branca": {
        20776, 20573, 20802, 29463, 29961, 29898, 20428, 32118, 26455, 20407, 20406,
        20817, 20804, 25433, 30509, 20819, 25212, 20011, 24713, 20460, 24715,
        29512, 32660, 20454, 20476, 29511, 30429, 26147, 32597, 32596, 26146,
        39980, 39979, 29615, 39982, 39983
    },
    "Persianas":          {25992, 26706},
    "Alvo K - Utilities": {32433},
    "Comunicação Visual": {20029},
    "CDU":                {24768},
    # 20103 → tratado separadamente (condicional por tipo_iniciativa)
    "Provisionamento":    {40062},
    # Montagem e Frete: pedidos independentes — classificados pelo material
    "Montagem":           {24935},
    "Frete":              {20272},
}

# Normalização de tipo_iniciativa — trata variações de maiúsculas/minúsculas
TIPO_NORMALIZACAO = {
    "lojas novas":      "Lojas Novas",
    "resize":           "Resize",
    "relocation":       "Relocation",
    "retrofit":         "Retrofit",
    "virada de bandeira": "Virada de Bandeira",  # preparado para uso futuro
}

def normalizar_tipo(valor):
    if pd.isna(valor):
        return "Lojas Novas"
    return TIPO_NORMALIZACAO.get(str(valor).strip().lower(), str(valor).strip())

# ======================================================
# HELPERS
# ======================================================

def classificar_por_texto(texto):
    t = str(texto).lower()
    if "luminaria" in t:
        return "Luminária"
    if "ar cond" in t or "cortina de ar" in t:
        return "Ar Condicionado - AC"
    if "serv" in t or "projeto" in t:
        return "Projetos"
    return None

# ======================================================
# CARGA SAP
# ======================================================

def load_real():
    df = pd.read_excel(FILE_REAL)
    df = df.rename(columns={
        "Ordem":                   "ordem",
        "Material":                "material",
        "Texto breve material":    "texto_breve",
        "Documento de compras":    "documento",
        "Valor/moeda objeto":      "valor"
    })
    df["base"] = "REAL"
    return df

def load_compromisso():
    df = pd.read_excel(FILE_COMP)
    df = df.rename(columns={
        "Ordem":                  "ordem",
        "Material":               "material",
        "Denominação":            "texto_breve",
        "Nº doc.de referência":   "documento",
        "Valor/moeda objeto":     "valor"
    })
    df["base"] = "COMPROMISSO"
    return df

# ======================================================
# PIPELINE PRINCIPAL
# ======================================================

def run_pipeline():
    print("🚀 Iniciando pipeline financeiro...")

    # 1. União SAP
    df = pd.concat([load_real(), load_compromisso()], ignore_index=True)

    df["valor"]    = pd.to_numeric(df["valor"],    errors="coerce").fillna(0)
    df["material"] = pd.to_numeric(df["material"], errors="coerce")
    df["documento"] = df["documento"].fillna("").astype(str)

    df["flag_pedido"] = df["documento"].str.startswith("4")
    df["flag_rc"]     = df["documento"].str.startswith("1")

    # 2. DIM_SITE — carrega aqui para usar na classificação do material 20103
    dim_site = pd.read_excel(FILE_SITE)
    # Lê filial se a coluna existir (coluna E), caso contrário cria vazia
    cols_base = ["ordem", "site", "responsavel", "tipo_iniciativa"]
    if "filial" in dim_site.columns:
        cols_base.append("filial")

    dim_site = dim_site[
        dim_site["responsavel"].notna() &
        (dim_site["responsavel"].astype(str).str.strip() != "")
    ][cols_base].copy()

    if "filial" not in dim_site.columns:
        dim_site["filial"] = ""

    # Converte float (ex: 7234.0) → int → string limpa (ex: "7234")
    def _limpar_filial(v):
        if pd.isna(v) or str(v).strip() == "":
            return ""
        try:
            return str(int(float(v)))
        except (ValueError, TypeError):
            return str(v).strip()

    dim_site["filial"] = dim_site["filial"].apply(_limpar_filial)

    dim_site["tipo_iniciativa"] = dim_site["tipo_iniciativa"].apply(normalizar_tipo)

    # Mapa ordem → tipo_iniciativa (para classificação condicional)
    mapa_tipo = dim_site.set_index("ordem")["tipo_iniciativa"].to_dict()

    # 3. Classificação de tarefa — regras fixas por material
    def classificar_material(row):
        mat = row["material"]

        # Material 20103: condicional por tipo de projeto
        if mat == 20103:
            tipo = mapa_tipo.get(row["ordem"], TIPO_LOJAS_NOVAS)
            return "Preservação" if tipo == TIPO_LOJAS_NOVAS else "Reforma de Loja"

        for tarefa, mats in REGRAS_MATERIAIS.items():
            if mat in mats:
                return tarefa
        return None

    df["tarefa"] = df.apply(classificar_material, axis=1)

    # Classificação por texto para linhas ainda sem tarefa
    mask = df["tarefa"].isna()
    df.loc[mask, "tarefa"] = df.loc[mask, "texto_breve"].apply(classificar_por_texto)

    # REAL sem documento → Capitalização
    df.loc[
        (df["base"] == "REAL") & (df["documento"].str.strip() == ""),
        "tarefa"
    ] = "Capitalização"

    # 4. Mobiliário e Gaveta
    #
    # Gaveta (24780) vem no mesmo pedido do Mobiliário mas é nota separada.
    # Por isso classificamos pelo material diretamente — não pelo pedido.
    #
    # Mobiliário: qualquer linha cujo material está na lista MATERIAIS_MOBILIARIO
    # Gaveta: material 24780 (mesmo que o pedido contenha também itens de Mobiliário)
    #
    # Ordem importa: Gaveta primeiro para não ser sobrescrita pelo Mobiliário

    # 4a. Gaveta — pelo material exato
    mask_gaveta = df["material"].isin(MATERIAIS_GAVETA) & df["tarefa"].isna()
    df.loc[mask_gaveta, "tarefa"] = "Gaveta"
    print(f"   ✅ Gaveta: {mask_gaveta.sum()} linhas")

    # 4b. Mobiliário — pela lista completa de materiais (exclui Gaveta já classificada)
    mask_mob = df["material"].isin(MATERIAIS_MOBILIARIO) & df["tarefa"].isna()
    df.loc[mask_mob, "tarefa"] = "Mobiliário"
    print(f"   ✅ Mobiliário: {mask_mob.sum()} linhas")

    # 4c. Linhas restantes sem tarefa dentro de pedidos de Mobiliário/Gaveta → Mobiliário
    pedidos_mob = set(
        df.loc[
            df["material"].isin(MATERIAIS_MOBILIARIO | MATERIAIS_GAVETA),
            "documento"
        ].unique()
    )
    pedidos_mob.discard("")
    mask_mob_resto = df["documento"].isin(pedidos_mob) & df["tarefa"].isna()
    df.loc[mask_mob_resto, "tarefa"] = "Mobiliário"
    print(f"   ✅ Mobiliário (linhas restantes do pedido): {mask_mob_resto.sum()}")

    # 5. Consolidação financeira
    financeiro = (
        df.groupby(["ordem", "tarefa"], as_index=False)
        .agg(
            REAL=("valor", lambda x: x[df.loc[x.index, "base"] == "REAL"].sum()),
            COMPROMISSO=("valor", lambda x: x[df.loc[x.index, "base"] == "COMPROMISSO"].sum()),
            flag_pedido=("flag_pedido", "max")
        )
        .fillna(0)
    )

    financeiro["COMPROMISSO_ABERTO"] = (
        financeiro["COMPROMISSO"] - financeiro["REAL"]
    ).clip(lower=0)

    # 6. MALHA COMPLETA — dinâmica por tipo_iniciativa
    #
    # Lojas Novas → usa TAREFAS_LOJAS_NOVAS (contém "Preservação", sem "Reforma de Loja")
    # Outros      → usa TAREFAS_OUTROS      (contém "Reforma de Loja", sem "Preservação")
    #
    malhas = []
    for _, row in dim_site.iterrows():
        tarefas = (
            TAREFAS_LOJAS_NOVAS
            if row["tipo_iniciativa"] == TIPO_LOJAS_NOVAS
            else TAREFAS_OUTROS
        )
        for tarefa in tarefas:
            malhas.append({
                "ordem":           row["ordem"],
                "site":            row["site"],
                "responsavel":     row["responsavel"],
                "tipo_iniciativa": row["tipo_iniciativa"],
                "filial":          row["filial"],
                "tarefa":          tarefa
            })

    malha = pd.DataFrame(malhas)

    fato = malha.merge(financeiro, on=["ordem", "tarefa"], how="left")

    fato[["REAL", "COMPROMISSO", "COMPROMISSO_ABERTO"]] = (
        fato[["REAL", "COMPROMISSO", "COMPROMISSO_ABERTO"]].fillna(0)
    )
    fato["flag_pedido"] = fato["flag_pedido"].fillna(False)

    # 7. STATUS FINAL
    fato["status"] = "Aguardando Requisição"

    fato.loc[
        (fato["REAL"] == 0) & (fato["COMPROMISSO"] > 0) & (~fato["flag_pedido"]),
        "status"
    ] = "Requisição Criada"

    fato.loc[
        (fato["REAL"] == 0) & (fato["COMPROMISSO"] > 0) & (fato["flag_pedido"]),
        "status"
    ] = "Pedido Criado"

    fato.loc[
        (fato["REAL"] > 0) & (fato["COMPROMISSO"] > 0),
        "status"
    ] = "Pgto Parcial"

    fato.loc[
        (fato["REAL"] > 0) & (fato["COMPROMISSO"] == 0),
        "status"
    ] = "Realizado"

    fato.loc[fato["tarefa"] == "Transferência", "status"] = "Controle Manual"

    # 8. Diagnóstico rápido
    print("\n📋 Diagnóstico — TER36-ZequinhaFre-PI:")
    amostra = fato[fato["site"].str.contains("TER36", na=False)][
        ["tarefa", "REAL", "COMPROMISSO", "COMPROMISSO_ABERTO", "status", "tipo_iniciativa"]
    ]
    if not amostra.empty:
        print(amostra.to_string(index=False))
    else:
        print("   (site não encontrado na amostra)")

    print("\n📊 Tipos de iniciativa na base:")
    print(fato["tipo_iniciativa"].value_counts().to_string())

    print("\n✅ Pipeline financeiro concluído\n")
    return fato

# ======================================================
# ENVIO PARA APPS SCRIPT
# ======================================================

def enviar(base):
    # Pré-popula dim_contratos com todas as combinações site+tarefa
    # Isso garante que a planilha sempre tenha as linhas corretas para preenchimento manual
    # Colunas manuais (fornecedor, contratado_manual, aditivo, avi) ficam vazias — não sobrescreve
    dim_pre = base[["site", "tarefa"]].copy()
    dim_pre["fornecedor"]        = ""
    dim_pre["contratado_manual"] = ""
    dim_pre["aditivo"]           = ""
    dim_pre["avi"]               = ""

    payload = {
        "metadata": {"gerado_em": datetime.now().isoformat()},
        "financeiro_consolidado": base[
            ["site", "filial", "tarefa", "REAL", "COMPROMISSO", "COMPROMISSO_ABERTO",
             "status", "responsavel", "tipo_iniciativa"]
        ].to_dict("records"),
        "dim_contratos_skeleton": dim_pre.to_dict("records")
    }

    r = requests.post(
        URL_APPS_SCRIPT,
        data=json.dumps(payload, default=str),
        headers={"Content-Type": "application/json"},
        timeout=180,
        verify=False
    )

    print("HTTP:", r.status_code)
    print("Resposta:", r.text)

# ======================================================
# MAIN
# ======================================================

if __name__ == "__main__":
    base = run_pipeline()
    enviar(base)

🚀 Iniciando pipeline financeiro...
   ✅ Gaveta: 18 linhas
   ✅ Mobiliário: 1159 linhas
   ✅ Mobiliário (linhas restantes do pedido): 30


C:\Users\126815\AppData\Local\Temp\ipykernel_26088\2049472687.py:317: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fato["flag_pedido"] = fato["flag_pedido"].fillna(False)



📋 Diagnóstico — TER36-ZequinhaFre-PI:
              tarefa      REAL  COMPROMISSO  COMPROMISSO_ABERTO                status tipo_iniciativa
         Construtora 981000.00      9455.00                0.00          Pgto Parcial     Lojas Novas
        Gerenciadora  33950.00         0.00                0.00             Realizado     Lojas Novas
  Alvo K - Utilities   2150.00     17377.38            15227.38          Pgto Parcial     Lojas Novas
           Persianas  17700.00         0.00                0.00             Realizado     Lojas Novas
           Luminária  23783.22         0.00                0.00             Realizado     Lojas Novas
Ar Condicionado - AC  48875.11         0.00                0.00             Realizado     Lojas Novas
        Linha Branca  15494.02     13953.58                0.00          Pgto Parcial     Lojas Novas
            Projetos  61960.00     33500.00                0.00          Pgto Parcial     Lojas Novas
  Comunicação Visual  95000.00         0.00

C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.google.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.googleusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


HTTP: 200
Resposta: {"status":"ok","rows":0}


In [1]:
import os
import math
import pandas as pd
import requests
import json
import warnings
from datetime import datetime

# ======================================================
# CONFIGURAÇÕES
# ======================================================

PASTA_BASE    = r"C:\Users\126815\OneDrive - paguemenos.com.br\ENGENHARIA - OBRAS - Documentos\BANCO DE DADOS - POWER BI\BI - OUTROS"
FILE_TICKETS  = os.path.join(PASTA_BASE, "BASE CONTROLE DE PAGAMENTOS.xlsx")
ABA_TICKETS   = "SOLICITAÇÃO DE PAGAMENTO"

URL_APPS_SCRIPT = "https://script.google.com/macros/s/AKfycbyeLMGPlTdfRNFBIb-Y_N_ZQgg6rz0r5IcW5QuNpWWSb0QRrCw13h6-WVDiBaN2jkCyjg/exec"

BATCH_SIZE = 2000

COLUNAS = [
    "FILIAL", "LOJA", "CNPJ", "COORDENADOR", "PROJETO", "SERVIÇO",
    "NOTA", "FORNECEDOR", "ARQUIVO", "ENVIAR COORD?", "cód forn",
    "ORDEM", "REQUISIÇÃO", "RC ENVIADA?", "VALOR RC", "VALOR A PAGAR",
    "VALOR BI", "STATUS RC", "PEDIDO", "ASSUNTO", "ID", "LINK ",
    "CHAMADO", "BASE_ANO", "Data_pgto_sap", "MIRO", "Abrir ticket?",
    "Status Result1", "Data criação ticket br", "Prazo", "Solicitado apoio?",
]

COLUNAS_DATA  = ["Data_pgto_sap", "Data criação ticket br"]
COLUNAS_VALOR = ["VALOR RC", "VALOR A PAGAR", "VALOR BI", "ARQUIVO"]


# ======================================================
# NORMALIZAÇÃO BASE_ANO
# ======================================================

def normalizar_base_ano(valor) -> str:
    try:
        if pd.isna(valor):
            return "2026"
    except (TypeError, ValueError):
        pass
    v = str(valor).strip()
    if v in ("", "nan", "NaN", "NaT", "0"):
        return "2026"
    if "2026" in v:
        return "2026"
    if "2025" in v:
        return "2025"
    return v


# ======================================================
# PIPELINE
# ======================================================

def run_pipeline_tickets() -> pd.DataFrame:
    print("🎫 Iniciando pipeline de tickets...")

    df = pd.read_excel(FILE_TICKETS, sheet_name=ABA_TICKETS, header=0, dtype=str)
    print(f"   📋 {len(df)} linhas lidas da base")

    # ── Filtro: linha ativa = tem FILIAL ou LOJA preenchida ──
    mask = (
        (df["FILIAL"].notna() & (df["FILIAL"].str.strip() != "")) |
        (df["LOJA"].notna()   & (df["LOJA"].str.strip()   != ""))
    )
    df = df[mask].copy()
    print(f"   🔍 Linhas ativas (com FILIAL ou LOJA): {len(df)}")

    df = df[COLUNAS].copy()

    # Limpa espaços
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    # Normaliza BASE_ANO
    df["BASE_ANO"] = df["BASE_ANO"].apply(normalizar_base_ano)
    anos = df["BASE_ANO"].value_counts().to_dict()
    print(f"   📅 BASE_ANO: {anos}")

    # Normaliza valores numéricos
    for col in COLUNAS_VALOR:
        if col in df.columns:
            df[col] = (
                df[col]
                .str.replace(",", ".", regex=False)
                .pipe(pd.to_numeric, errors="coerce")
            )

    # Normaliza datas
    for col in COLUNAS_DATA:
        if col in df.columns:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                parsed = pd.to_datetime(df[col], format="mixed", dayfirst=True, errors="coerce")
            df[col] = parsed.dt.strftime("%d/%m/%Y").where(parsed.notna(), other="")

    df = df.fillna("")
    print(f"   ✅ {len(df)} linhas prontas para envio")
    return df


# ======================================================
# ENVIO COM PAGINAÇÃO
# ======================================================

def _post(payload: dict) -> dict:
    r = requests.post(
        URL_APPS_SCRIPT,
        data=json.dumps(payload, default=str),
        headers={"Content-Type": "application/json"},
        timeout=300,
        verify=False,
    )
    try:
        return r.json()
    except Exception:
        return {"error": r.text}


def enviar_tickets(df: pd.DataFrame):
    registros = df.to_dict("records")
    total     = len(registros)
    n_batches = math.ceil(total / BATCH_SIZE)

    print(f"📤 Enviando {total} tickets em {n_batches} lote(s) de até {BATCH_SIZE}...")

    for i in range(n_batches):
        inicio = i * BATCH_SIZE
        fim    = min(inicio + BATCH_SIZE, total)
        lote   = registros[inicio:fim]

        payload = {
            "metadata": {
                "gerado_em":     datetime.now().isoformat(),
                "origem":        "pipeline_tickets",
                "batch":         i + 1,
                "total_batches": n_batches,
                "modo":          "reset" if i == 0 else "append",
            },
            "controle_tickets": lote,
        }

        resp = _post(payload)

        if "error" in resp:
            print(f"   ❌ Lote {i+1}/{n_batches} ERRO: {resp['error']}")
            break

        status  = resp.get("status") or ("ok" if resp.get("ok") else "?")
        tickets = resp.get("tickets", resp.get("rows", "?"))
        modo    = resp.get("modo", "?")
        print(f"   ✅ Lote {i+1}/{n_batches} ({len(lote)} linhas) → status={status} | gravados={tickets} | modo={modo}")

    print("✅ Envio de tickets concluído.")


# ======================================================
# MAIN
# ======================================================

if __name__ == "__main__":
    df_tickets = run_pipeline_tickets()
    enviar_tickets(df_tickets)

🎫 Iniciando pipeline de tickets...
   📋 13367 linhas lidas da base
   🔍 Linhas ativas (com FILIAL ou LOJA): 2840
   📅 BASE_ANO: {'2025': 2305, '2026': 535}
   ✅ 2840 linhas prontas para envio
📤 Enviando 2840 tickets em 2 lote(s) de até 2000...


C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.google.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.googleusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   ✅ Lote 1/2 (2000 linhas) → status=ok | gravados=2000 | modo=reset


C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.google.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.googleusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


   ✅ Lote 2/2 (840 linhas) → status=ok | gravados=840 | modo=append
✅ Envio de tickets concluído.


In [19]:
import os
import pandas as pd
import requests
import json
from datetime import datetime

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = os.path.join(
    r"C:\Users\126815\OneDrive - paguemenos.com.br",
    r"ENGENHARIA - OBRAS - Documentos",
    r"BANCO DE DADOS - PLATAFORMA FINANCEIRA",
    r"PROJETO_PLATAFORMA_FINANCEIRA"
)

FILE_SITE = os.path.join(
    BASE_PATH,
    "data",
    "dimensions",
    "dim_site.xlsx"
)

URL_APPS_SCRIPT = (
    "https://script.google.com/macros/s/"
    "AKfycbyeLMGPlTdfRNFBIb-Y_N_ZQgg6rz0r5IcW5QuNpWWSb0QRrCw13h6-WVDiBaN2jkCyjg"
    "/exec"
)

# ======================================================
# PIPELINE
# ======================================================

def run_pipeline_utilities() -> pd.DataFrame:
    print("⚡ Iniciando pipeline de utilities...")

    dim = pd.read_excel(FILE_SITE, dtype=str)

    # Filtra só Lojas Novas
    ln = dim[dim["tipo_iniciativa"].str.strip() == "Lojas Novas"].copy()
    print(f"   📋 {len(ln)} sites de Lojas Novas encontrados")

    records = []

    for _, row in ln.iterrows():
        ordem = str(row.get("ordem", "") or "").strip()
        if ordem in ("0", "nan", ""):
            ordem = ""

        filial = str(row.get("filial", "") or "").strip()
        if filial.endswith(".0"):
            filial = filial[:-2]
        if filial in ("nan", ""):
            filial = ""

        records.append({
            "ordem": ordem,
            "site": str(row.get("site", "") or "").strip(),
            "filial": filial,
            "coord": str(row.get("responsavel", "") or "").strip(),

            # ✅ MAPEAMENTO DIRETO DA dim_site
            # inauguração → dt_abertura
            # status da obra → status_loja
            "dt_abertura": str(row.get("dt_abertura", "") or "").strip(),
"status_loja": str(row.get("status_loja", "") or "").strip(),

            # Campos operacionais
            "result": "PENDENTE",
            "ticket_agua": "",
            "data_ticket_agua": "",
            "status_ticket_agua": "AGUARDANDO DADOS",
            "obs_agua": "",
            "ticket_energia": "",
            "data_ticket_energia": "",
            "status_ticket_energia": "AGUARDANDO DADOS",
            "obs_energia": "",
        })

    df = pd.DataFrame(records)
    print(f"   ✅ {len(df)} linhas prontas")
    return df

# ======================================================
# ENVIO
# ======================================================

def _post(payload: dict) -> dict:
    r = requests.post(
        URL_APPS_SCRIPT,
        data=json.dumps(payload, default=str),
        headers={"Content-Type": "application/json"},
        timeout=300,
        verify=False,
    )
    try:
        return r.json()
    except Exception:
        return {"error": r.text[:300]}

def enviar_utilities(df: pd.DataFrame):
    print("📤 Enviando utilities para o Google Sheets...")
    payload = {
        "metadata": {
            "gerado_em": datetime.now().isoformat(),
            "origem": "pipeline_utilities",
        },
        "controle_utilities": df.to_dict("records"),
    }
    resp = _post(payload)
    print("Resposta:", resp)

# ======================================================
# MAIN
# ======================================================

if __name__ == "__main__":
    df = run_pipeline_utilities()
    enviar_utilities(df)

⚡ Iniciando pipeline de utilities...
   📋 55 sites de Lojas Novas encontrados
   ✅ 55 linhas prontas
📤 Enviando utilities para o Google Sheets...


C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.google.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\126815\AppData\Roaming\Python\Python313\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'script.googleusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Resposta: {'status': 'ok', 'utilities': 55, 'versao': 'UTILITIES v6.1'}
